In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mne

In [ ]:
import os
dataset_path = "/content/drive/MyDrive/ds003844"
print(os.listdir(dataset_path))

In [ ]:
import mne
import pandas as pd
vhdr_files = []
channels_files = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".vhdr"):
            vhdr_files.append(os.path.join(root, file))
        elif file.endswith("_channels.tsv"):
            channels_files.append(os.path.join(root, file))
vhdr_files = sorted(vhdr_files)
channels_files = sorted(channels_files)

In [ ]:
sfreqs = []
for vhdr in vhdr_files:
    raw = mne.io.read_raw_brainvision(vhdr, preload=False, verbose=False)
    sfreqs.append({
        "recording": os.path.basename(vhdr),
        "sfreq": raw.info["sfreq"]
    })
sfreq_df = pd.DataFrame(sfreqs)
print(sfreq_df)

In [ ]:
vhdr_2048 = []
channels_2048 = []
for vhdr, ch_file in zip(sorted(vhdr_files), sorted(channels_files)):
    raw = mne.io.read_raw_brainvision(
        vhdr,
        preload=False,
        verbose=False
    )
    if raw.info["sfreq"] >= 2048:
        vhdr_2048.append(vhdr)
        channels_2048.append(ch_file)
print("Recordings kept:", len(vhdr_2048))
print("Recordings removed:", len(vhdr_files) - len(vhdr_2048))

In [ ]:
raw.plot_psd(fmax=100)

In [ ]:
results = []
processed_path = "/content/drive/MyDrive/ds003844_processed"
os.makedirs(processed_path, exist_ok=True)
for vhdr, ch_file in zip(sorted(vhdr_2048), sorted(channels_2048)):
    raw = mne.io.read_raw_brainvision(vhdr, preload=True, verbose=False)
    before = len(raw.ch_names)
    ch_df = pd.read_csv(ch_file, sep="\t")
    bad = ch_df[ch_df["status"] == "bad"]["name"].tolist()
    raw.drop_channels([ch for ch in bad if ch in raw.ch_names])
    raw.notch_filter(freqs=[50], verbose=False)
    raw.resample(1024)
    after = len(raw.ch_names)
    results.append([os.path.basename(vhdr),before, after, len(bad)])
    save_name = os.path.basename(vhdr).replace(".vhdr","_processed_raw.fif")
    raw.save(os.path.join(processed_path, save_name),overwrite=True)
    print("Saved:", save_name)
results_df = pd.DataFrame(results,columns=["recording","before", "after", "removed"])
results_df.head(38)

In [ ]:
def get_label(filename):
    if "SITUATION1" in filename:
        return "PRE"
    elif "SITUATION2" in filename:
        return "INTER"
    elif "SITUATION3" in filename:
        return "POST"
    elif "SITUATION4" in filename:
        return "POST+"

In [ ]:
all_windows = []
all_labels = []
sequence_lengths = []
recording_names = []
channel_counts = []
window_size = 5 * 1024
step_size = window_size // 2
# Sort for deterministic order
files = sorted([f for f in os.listdir(processed_path) if f.endswith(".fif")])
for file in files:
    raw = mne.io.read_raw_fif(
        os.path.join(processed_path, file),
        preload=True,
        verbose=False
    )
    data = raw.get_data()
    label = get_label(file)
    n_windows = 0
    for start in range(
        0,
        data.shape[1] - window_size + 1,
        step_size
    ):
        end = start + window_size
        window = data[:, start:end]
        all_windows.append(window)
        all_labels.append(label)
        n_windows += 1
    # Store metadata for this recording
    sequence_lengths.append(n_windows)
    recording_names.append(file)
    channel_counts.append(data.shape[0])
print("Total recordings :", len(sequence_lengths))
print("Total windows    :", len(all_windows))
print("Total labels     :", len(all_labels))
print("Max channels     :", max(channel_counts))
print("\nFirst five recordings:")
for i in range(min(5, len(recording_names))):
    print(
        f"{i+1:2d}. {recording_names[i]} | "
        f"Windows: {sequence_lengths[i]} | "
        f"Channels: {channel_counts[i]}"
    )

In [ ]:
import pickle
metadata = {
    "recording_names": recording_names,
    "sequence_lengths": sequence_lengths,
    "channel_counts": channel_counts
}
with open(
    "/content/drive/MyDrive/ds003844_processed/recording_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)
print("Metadata saved successfully.")

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import numpy as np
window = all_windows[0]
n_channels = window.shape[0]
mi_matrix = np.zeros((n_channels, n_channels))
for i in range(n_channels):
    for j in range(i + 1, n_channels):
        mi = mutual_info_regression(
            window[i].reshape(-1, 1),
            window[j],
            random_state=42
        )[0]
        mi_matrix[i, j] = mi
        mi_matrix[j, i] = mi
print(mi_matrix.shape)

In [ ]:
print(mi_matrix[:26, :26])

In [ ]:
upper = mi_matrix[np.triu_indices_from(mi_matrix, k=1)]
print("Min:", upper.min())
print("Max:", upper.max())
print("Mean:", upper.mean())
print("Median:", np.median(upper))

In [ ]:
import matplotlib.pyplot as plt
plt.hist(upper, bins=30)
plt.xlabel("Mutual Information")
plt.ylabel("Count")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.imshow(mi_matrix)
plt.colorbar(label="Mutual Information")
plt.title("Adaptive Brain Graph - Window 0")
plt.xlabel("Channel")
plt.ylabel("Channel")
plt.show()
mi_matrix0 = mi_matrix.copy()

In [ ]:
import os
import pickle
import numpy as np
from joblib import Parallel, delayed
from sklearn.feature_selection import mutual_info_regression

In [ ]:
def compute_mi_graph(window):
    n_channels = window.shape[0]
    mi_matrix = np.zeros((n_channels, n_channels), dtype=np.float32)
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            mi = mutual_info_regression(
                window[i].reshape(-1,1),
                window[j],
                random_state=42
            )[0]
            mi_matrix[i,j] = mi
            mi_matrix[j,i] = mi
    return mi_matrix

In [ ]:
save_folder = "/content/drive/MyDrive/ds003844_processed/adaptive_graph_batches"
os.makedirs(save_folder, exist_ok=True)

In [ ]:
batch_size = 100
total_windows = len(all_windows)
for start in range(0, total_windows, batch_size):
    end = min(start + batch_size, total_windows)
    print(f"\nProcessing windows {start} to {end-1}")
    batch_graphs = Parallel(
        n_jobs=-1,
        backend="loky",
        verbose=10
    )(
        delayed(compute_mi_graph)(window)
        for window in all_windows[start:end]
    )
    batch_file = os.path.join(
        save_folder,
        f"graphs_{start}_{end-1}.pkl"
    )
    with open(batch_file, "wb") as f:
        pickle.dump(batch_graphs, f)
    print(f"Saved {batch_file}")

In [ ]:
import glob
graph_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/ds003844_processed/adaptive_graph_batches/*.pkl"
    )
)
graphs = []
for file in graph_files:
    with open(file, "rb") as f:
        graphs.extend(pickle.load(f))
print(len(graphs))